# Lab 12 - Inference Performance Metrics on Real Hardware

**Textbook:** Chip Huyen, *AI Engineering*, Chapter 9 "Inference Optimization", first half: "Understanding Inference Optimization" (Inference Overview, Inference Performance Metrics, AI Accelerators).
**Environment:** cluster notebook (your local Docker JupyterLab on the KU VPN).
**Time:** about 4-5 hours, including waiting for measurements.

Larkspur's Service Desk Manager wants to know whether the help-desk assistant will feel fast enough for technicians on a busy Monday morning, and how many technicians the course cluster could serve. Nobody can answer that from a spec sheet. You'll answer it the way an AI engineer does: by **measuring** time to first token (TTFT), time per output token (TPOT), throughput and goodput on the real hardware (three Mac Studio M3 Ultra machines running oMLX, the program that runs the models on each Mac, behind a LiteLLM gateway, the server that receives your requests and passes them on).

### Before you start
- Connect to the **KU VPN** and start your course container (`docker compose up`).
- Read Chapter 9 from the start through "AI Accelerators" (the part called "Understanding Inference Optimization"). Keep it open; the analysis asks for section names.
- **The cluster is shared.** Your numbers depend on what classmates are doing at the same moment. That is not a bug in your lab: it is part of what you'll analyze. Don't re-run the big cells over and over; this notebook sends about 110 requests in total (126 if you run the optional 8-way test).
- This notebook uses `await` at the top level of cells. `await` means "wait for this request to finish, but let other requests run meanwhile"; it is how one notebook keeps several requests going at once. That works in Jupyter; just run cells in order.

## Words you'll meet in this lab

You don't need to memorize these. Each one is explained again where it first matters; come back here if a word stops you.

| Term | What it means | Why it matters |
|---|---|---|
| **Inference** | Running a trained model to get an answer (from Lab 01). | This whole lab measures how fast inference is. |
| **Latency** | How long one user waits for one request, in seconds. | It decides whether the assistant feels fast. |
| **Streaming** | The server sends the answer piece by piece as it is written, instead of all at once at the end. | It lets you time the first token separately from the rest. |
| **TTFT (time to first token)** | Seconds from sending a request until the first piece of the answer arrives. | What a user feels as "is it doing anything?" |
| **TPOT (time per output token)** | Seconds between one output token and the next, once the answer is flowing. | Decides whether text appears faster than people read. |
| **Prefill / decode** | The two phases of a request. Prefill: the model reads the whole prompt in one go. Decode: it writes the answer one token at a time. | Prefill mostly sets TTFT; decode sets TPOT. |
| **Compute-bound / memory-bandwidth-bound** | Whether a job is limited by how much math the chip can do, or by how fast it can read data from memory. | Prefill is compute-bound and decode is bandwidth-bound, which explains your results. |
| **Memory bandwidth** | How many gigabytes per second a chip can read from its memory (from Lab 02). | It sets the speed limit for decode. |
| **MBU (model bandwidth utilization)** | The share of the chip's peak memory bandwidth a model actually uses while decoding. | Tells you how close you are to the hardware's limit. |
| **Percentile (p50, p90)** | p50 (the median) is the middle request; p90 means 90% of requests were at least this fast. | Averages hide slow requests; p90 shows what the unlucky users get. |
| **Concurrency** | How many requests are in progress at the same moment. | More users at once means more concurrency. |
| **Throughput** | How much work the whole system delivers: output tokens (or requests) per second, across all users. | A busy service cares about throughput; one user cares about latency. |
| **Batching (continuous batching)** | Processing several requests together in one pass over the weights. In continuous batching, requests join and leave the batch as they arrive and finish. | It is why throughput can rise when many users arrive at once. |
| **SLO (service-level objective)** | Your target for "fast enough", for example TTFT under 2 s. | You decide it, and it defines goodput. |
| **Goodput** | Requests per second that meet the SLO. | Fast-but-late answers don't count. |
| **Prompt cache** | The server keeps the work it did on a recent prompt and reuses it if a new prompt starts the same way. | It can make timing look better than it is, so this lab switches it off on purpose. |
| **oMLX, LiteLLM** | oMLX is the program on each Mac Studio that runs the models. LiteLLM is the gateway software that receives your requests and passes them to a Mac. | Both add their own small delays to what you measure. |
| **Rate limit, HTTP 429** | Your key may send only so many requests per minute and run only 4 at once. Extra requests are refused with error code 429 ("Too Many Requests"). | The notebook pauses between requests to stay under the limit. |
| **`async` / `await`** | A Python feature that lets one program wait for several network requests at the same time instead of one after another. | You need it to send requests in parallel in Part B. |
| **Little's law** | A general rule from queueing theory (not from the textbook): average number of requests in progress = arrival rate x average latency. B4 explains it with a help-desk phone example. | You use it to estimate how many technicians the cluster can serve. |

## Setup

Run this cell. Besides the usual client, it creates an **async client** (`aclient`), which can have several requests in flight at the same time, and it records when you started, because your numbers depend on what else the cluster was doing at that moment.

In [ ]:
import asyncio, time, uuid, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openai import AsyncOpenAI
from itec632 import get_client, MODELS, NO_THINK, settings
from itec632.corpus import load_docs
from itec632.grading import LabChecker

ck = LabChecker("lab12", part="cluster")
client = get_client()                         # stops with a clear message if your .env is not set
base_url, api_key = settings()
# Async client for streaming and for parallel requests.
# max_retries=0: a silent automatic retry would hide inside your timing and corrupt it.
aclient = AsyncOpenAI(base_url=base_url, api_key=api_key, timeout=300, max_retries=0)

MOE, DENSE, SMALL = MODELS["main"], MODELS["dense"], MODELS["small"]
CHAT_MODELS = [MOE, DENSE, SMALL]
COLORS = {MOE: "#2a78d6", DENSE: "#eb6834", SMALL: "#1baf7a"}   # same color per model in every chart
PAUSE_S = 1.5        # pause between sequential requests (stays under the 60 requests/minute key limit)

available = [m.id for m in client.models.list().data]
print("Models on the gateway:", available)
ck.check("connected", all(m in available for m in CHAT_MODELS), 1,
         "Could not see qwen-moe, qwen-dense and qwen-small. Check the VPN and your .env, then re-run.")

RUN_STARTED_UTC = dt.datetime.now(dt.timezone.utc)
ck.record("run_started_utc", RUN_STARTED_UTC.strftime("%Y-%m-%d %H:%M"))
print("Started (UTC):", RUN_STARTED_UTC.strftime("%A %Y-%m-%d %H:%M"),
      "| Central Time is UTC-6 in winter (UTC-5 in summer). Note your local time for the analysis.")

## Part A - Guided practice

### A1. Anatomy of a streamed response: TTFT and TPOT

Chapter 9 ("Inference Performance Metrics") splits the latency of one request into two parts:

- **TTFT (time to first token):** how long until the first token arrives. This covers network time, queueing, and **prefill**: the model processing your whole prompt in one parallel pass.
- **TPOT (time per output token):** how long each further token takes during **decode**, when the model generates one token at a time.

So total latency is about `TTFT + TPOT x (number of output tokens - 1)`. (The chapter writes it with the token count; the `- 1` is the exact number of gaps after the first token.) TTFT is what a user feels as "is it doing anything?"; TPOT decides whether the text streams faster than people read.

You need **streaming** to see these: a normal (non-streamed) call only tells you the total. Two details make the timing correct:

1. **Chunks are not tokens.** The server may pack several tokens into one network chunk (and models with multi-token prediction, from Lab 02, can emit more than one token per step). So we count tokens with `usage.completion_tokens`, which the server sends in a final chunk when we pass `stream_options={"include_usage": True}`.
2. **The usage chunk can be missing** (an interrupted stream, or a server that ignores the option). The code below then falls back to an estimate (about 4 characters per token) and sets `usage_estimated=True`.

`measure_stream()` and the small function it calls at the end, `compute_metrics()`, are written for you. Read `compute_metrics()`: TTFT is the time until the first token, total is the time until the stream ends, TPOT is the decode time (after the first token) divided by the number of gaps between tokens, and `out_tok_per_s` is the output tokens divided by the total time. Your job here is to **predict, measure, and compare**.

**TODO (A1):** before you run the test request, write your guesses for `qwen-moe` in `PREDICT_A1`: how many seconds until the first token, and how many tokens per second it will write once it has started. (For scale: a fast reader reads about 5-8 words per second, roughly 7-10 tokens per second.) Any honest guess is fine; the point is to compare it with what you measure.

In [ ]:
def compute_metrics(t_start, t_first, t_end, n_out):
    """Turn the timestamps of one streamed request into latency metrics.

    t_start: perf_counter() just before the request was sent
    t_first: perf_counter() when the first token arrived
    t_end:   perf_counter() when the stream finished
    n_out:   number of output tokens (usage.completion_tokens)
    """
    ttft = t_first - t_start
    total = t_end - t_start
    tpot = (t_end - t_first) / (n_out - 1) if n_out and n_out > 1 else None
    out_tok_per_s = n_out / total if total > 0 else None
    return {"ttft_s": ttft, "tpot_s": tpot, "total_s": total, "out_tok_per_s": out_tok_per_s}

In [ ]:
BUDGET = {"sent": 0}   # counts every request this notebook sends (the cluster is shared!)


def _estimate_tokens(text):
    """Rough fallback: about 4 characters per token for English text."""
    return max(1, round(len(text or "") / 4))


async def measure_stream(model, messages, max_tokens=200, include_usage=True, extra_body=None):
    """Send ONE streamed chat request and time it. Returns a dict (one row of results).

    Timing uses time.perf_counter(), a monotonic high-resolution clock made for
    measuring intervals (time.time() can jump if the system clock is adjusted).
    """
    rec = {"model": model, "max_tokens": max_tokens, "ok": False, "error": None,
           "ttft_s": None, "tpot_s": None, "total_s": None, "out_tok_per_s": None,
           "prompt_tokens": None, "completion_tokens": None, "cached_tokens": None,
           "usage_estimated": False, "est_completion_tokens": None, "n_chunks": 0,
           "finish_reason": None, "backend": None, "text": "", "chunk_times": [],
           "sent_utc": dt.datetime.now(dt.timezone.utc).strftime("%H:%M:%S")}
    kwargs = dict(model=model, messages=messages, max_tokens=max_tokens, stream=True,
                  extra_body=NO_THINK if extra_body is None else extra_body)
    if include_usage:
        # Ask the server to append one final chunk with exact token counts.
        kwargs["stream_options"] = {"include_usage": True}
    BUDGET["sent"] += 1
    usage, parts, t_first = None, [], None
    t_start = time.perf_counter()
    try:
        # with_raw_response gives us the HTTP headers too (which backend Mac served us).
        raw = await aclient.chat.completions.with_raw_response.create(**kwargs)
        rec["backend"] = raw.headers.get("x-litellm-model-id") or raw.headers.get("x-litellm-model-api-base")
        async for chunk in raw.parse():
            now = time.perf_counter()
            if getattr(chunk, "usage", None):
                usage = chunk.usage                      # the final usage chunk
            for choice in chunk.choices or []:          # the usage chunk has no choices
                delta = choice.delta
                piece = getattr(delta, "content", None) or ""
                thought = getattr(delta, "reasoning_content", None) or ""
                if piece or thought:
                    if t_first is None:
                        t_first = now                    # first token of any kind
                    rec["chunk_times"].append(round(now - t_start, 4))
                    parts.append(piece)
                if choice.finish_reason:
                    rec["finish_reason"] = choice.finish_reason
        t_end = time.perf_counter()
    except Exception as e:                               # e.g. RateLimitError (429), timeout
        rec["total_s"] = time.perf_counter() - t_start
        rec["error"] = f"{type(e).__name__}: {e}"[:200]
        return rec
    text = "".join(parts)
    rec["text"], rec["n_chunks"] = text, len(rec["chunk_times"])
    rec["est_completion_tokens"] = _estimate_tokens(text)
    if usage is not None and usage.completion_tokens:
        rec["prompt_tokens"] = usage.prompt_tokens
        rec["completion_tokens"] = usage.completion_tokens
        details = getattr(usage, "prompt_tokens_details", None)
        rec["cached_tokens"] = getattr(details, "cached_tokens", None) if details else None
    else:
        # No usage chunk (stream cut off, or a server that ignores stream_options):
        # fall back to an estimate and flag it so you know the number is approximate.
        rec["usage_estimated"] = True
        rec["prompt_tokens"] = sum(_estimate_tokens(m["content"]) for m in messages)
        rec["completion_tokens"] = rec["est_completion_tokens"]
    if t_first is None:
        rec["error"] = "stream ended without any content"
        rec["total_s"] = t_end - t_start
        return rec
    rec.update(compute_metrics(t_start, t_first, t_end, rec["completion_tokens"]))
    rec["ok"] = True
    return rec


def p50_p90(df, by, cols):
    """Median (p50) and 90th percentile (p90) of each column, per group. Failed requests are skipped."""
    rows = []
    ok = df[df["ok"] == True] if len(df) else df  # noqa: E712
    for key, g in ok.groupby(by):
        row = dict(zip(by, key)) if isinstance(by, list) else {by: key}
        row["n"] = len(g)
        for c in cols:
            s = pd.to_numeric(g[c], errors="coerce").dropna()
            row[f"{c}_p50"] = s.median() if len(s) else np.nan
            row[f"{c}_p90"] = s.quantile(0.9) if len(s) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def label_backends(df):
    """Replace long backend ids from the gateway headers with short labels: mac-A, mac-B, ..."""
    names = {}
    for b in df["backend"]:
        if b and b not in names:
            names[b] = f"mac-{chr(65 + len(names))}"
    return df["backend"].map(lambda b: names.get(b, "unknown"))


def nonce():
    """A short random tag. Putting it at the START of a prompt makes the prompt unique,
    so the server cannot reuse a cached prefix from an earlier request."""
    return uuid.uuid4().hex[:8]

In [ ]:
# TODO: your guesses for qwen-moe, BEFORE you run the next cell (numbers)
PREDICT_A1 = {"ttft_s": None, "decode_tok_s": None}

In [ ]:
# One request to try it out. Read every field: which ones come from the server, which from your clock?
QUICK = [{"role": "user", "content": "In about 100 words, explain what a help-desk ticket priority is."}]
r = await measure_stream(MOE, QUICK, max_tokens=150)
show = {k: v for k, v in r.items() if k not in ("text", "chunk_times")}
print(pd.Series(show).to_string())
print("\nAnswer starts:", r["text"][:200].replace("\n", " "))

**Reading the output:** each line is one field of the result. `ttft_s`, `tpot_s` and `total_s` are the timings in seconds, and `out_tok_per_s` is output tokens per second for the whole request. `prompt_tokens` and `completion_tokens` are the input and output token counts; `cached_tokens` is how many prompt tokens the server reused from its prompt cache (empty if it doesn't say). `usage_estimated` is `True` when the token count is the 4-characters-per-token guess. `n_chunks` is how many network pieces arrived. `finish_reason` is why the answer stopped (`stop`: the model finished; `length`: it hit `max_tokens`). `backend` identifies which Mac answered, and `sent_utc` is when the request was sent. `None` means a value was not available.

In [ ]:
# Your guesses vs the measurement. The first line shows how the three timings fit together:
# total ~= TTFT + TPOT x (n - 1)  (with TPOT defined this way it holds by arithmetic)
ok_measured = False
if r["ok"] and r["tpot_s"] and r["total_s"]:
    from_parts = r["ttft_s"] + r["tpot_s"] * (r["completion_tokens"] - 1)
    ok_measured = r["ttft_s"] > 0 and r["tpot_s"] > 0 and abs(from_parts - r["total_s"]) < 0.05 * r["total_s"] + 0.01
    print(f"measured total {r['total_s']:.3f} s, TTFT + TPOT x (n-1) = {from_parts:.3f} s")
    print(f"TTFT:   you guessed {PREDICT_A1['ttft_s']} s, measured {r['ttft_s']:.2f} s")
    print(f"decode: you guessed {PREDICT_A1['decode_tok_s']} tokens/s, measured {1 / r['tpot_s']:.1f} tokens/s (1 / TPOT)")
ck.check("a1_prediction", all(isinstance(v, (int, float)) and v > 0 for v in PREDICT_A1.values()), 1,
         "Put a positive number for both guesses in PREDICT_A1 (seconds to first token, tokens per second).")
ck.check("a1_measured", ok_measured, 2,
         "The test request did not return usable timings. Check the VPN and your .env, then re-run the test request.")
ck.record("a1_predict_ttft_s", PREDICT_A1.get("ttft_s"))
ck.record("a1_predict_decode_tok_s", PREDICT_A1.get("decode_tok_s"));

In [ ]:
# Picture of one stream: when did each chunk arrive? The flat start is prefill (TTFT); the slope is decode.
if r["ok"]:
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.plot(r["chunk_times"], range(1, len(r["chunk_times"]) + 1), color=COLORS[MOE], lw=2)
    ax.axvline(r["ttft_s"], color="#888888", lw=1, ls="--")
    ax.text(r["ttft_s"], 1, "  TTFT", color="#555555", va="bottom")
    ax.set_xlabel("seconds since request was sent"); ax.set_ylabel("chunks received")
    ax.set_title(f"One streamed response from {MOE}"); ax.grid(alpha=0.3); ax.set_xlim(left=0)
    plt.show()
    print(f"chunks: {r['n_chunks']}, tokens (usage): {r['completion_tokens']}, 4-chars-per-token estimate: {r['est_completion_tokens']}")

**Reading the plot:** across is time since the request was sent; up is how many chunks have arrived so far. The flat stretch at the start (up to the dashed TTFT line) is waiting: network, queue and prefill. After that the line climbs as the answer streams in; a steeper climb means faster decode (smaller TPOT).

### A2. Repeat, then report median and p90

One measurement is an anecdote. Latency on a shared cluster varies with the network, with what classmates send, and (for `qwen-moe`) with which of the two Macs the gateway picks. Chapter 9 recommends looking at latency as a **distribution**, with percentiles such as p50 (median) and p90, not as an average that one slow request can distort. (p90 is the value that 90% of requests beat: it shows what the slower requests experience.)

The loop below sends the same long-answer prompt to all three models. It **interleaves** the models (moe, dense, small, moe, dense, ...) so a burst of classmate traffic affects all models equally instead of landing on just one. `max_tokens=200` with a "write about 300 words" prompt means nearly every answer runs to exactly 200 tokens, so the models are compared on equal work.

**TODO:** inside the loop, call `measure_stream` and keep the result.

In [ ]:
PROMPT_A = [
    {"role": "system", "content": "You are Larkspur Ridge IT's help-desk assistant."},
    {"role": "user", "content": "Write a detailed step-by-step guide (about 300 words) for a Tier 1 technician "
                                "helping a user whose Outlook keeps asking for their password."},
]
N_REPS = 6
MAX_TOKENS_A = 200

# Warm-up: one untimed request per model (opens connections; the models are already loaded).
for m in CHAT_MODELS:
    await measure_stream(m, QUICK, max_tokens=20)

part_a_records = []
for rep in range(N_REPS):
    for model in CHAT_MODELS:
        # TODO: rec = await measure_stream(...) for this model, PROMPT_A and MAX_TOKENS_A
        rec = None
        if rec is not None:
            rec["rep"] = rep
            part_a_records.append(rec)
        await asyncio.sleep(PAUSE_S)
    print(f"rep {rep + 1}/{N_REPS} done")

df_a = pd.DataFrame(part_a_records)
print("errors:", int((~df_a["ok"]).sum()) if len(df_a) else "no data")

In [ ]:
METRICS = ["ttft_s", "tpot_s", "total_s", "out_tok_per_s"]
summary_a = p50_p90(df_a, "model", METRICS) if len(df_a) else pd.DataFrame()
if len(summary_a):
    summary_a["decode_tok_per_s"] = 1 / summary_a["tpot_s_p50"]
    display(summary_a.round(3))
    if df_a["usage_estimated"].any():
        print("Note: some token counts are estimates (no usage chunk arrived).")

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    for ax, col, label in [(axes[0], "ttft_s", "TTFT (s)"), (axes[1], "tpot_s", "TPOT (s per token)")]:
        data = [pd.to_numeric(df_a[(df_a.model == m) & df_a.ok][col], errors="coerce").dropna() for m in CHAT_MODELS]
        for i, (m, d) in enumerate(zip(CHAT_MODELS, data)):
            ax.scatter([i] * len(d), d, color=COLORS[m], s=40, alpha=0.8)
            if len(d):
                ax.hlines(d.median(), i - 0.25, i + 0.25, color="#333333", lw=2)
        ax.set_xticks(range(len(CHAT_MODELS)), CHAT_MODELS); ax.set_ylabel(label); ax.grid(alpha=0.3, axis="y")
    axes[0].set_title("Each dot is one request; bar = median")
    plt.tight_layout(); plt.show()
else:
    print("Complete the TODO above first.")

**Reading the table and plot:** one row per model. `n` is how many requests succeeded. For each timing (`ttft_s`, `tpot_s`, `total_s`, `out_tok_per_s`) there is a `_p50` column (the median) and a `_p90` column (90% of requests were at or below this). `decode_tok_per_s` is `1 / median TPOT`: how many tokens per second the model writes once it has started. In the plots, each dot is one request and the dark bar is the median; a wide spread of dots means the timing varied a lot from request to request.

In [ ]:
def p50(model, col):
    row = summary_a[summary_a["model"] == model] if len(summary_a) else []
    return float(row[f"{col}_p50"].iloc[0]) if len(row) and pd.notna(row[f"{col}_p50"].iloc[0]) else None

counts = df_a[df_a["ok"] == True].groupby("model").size() if len(df_a) else pd.Series(dtype=int)  # noqa: E712
ck.check("part_a_all_models", all(counts.get(m, 0) >= 5 for m in CHAT_MODELS), 3,
         "Need at least 5 successful requests per model. Finish the TODO; if you see errors, read df_a['error'].")
tp_moe, tp_dense = p50(MOE, "tpot_s"), p50(DENSE, "tpot_s")
# Not graded (it depends on the cluster, not on your code): did qwen-dense decode more slowly, as expected?
ck.record("dense_decodes_slower", bool(tp_moe and tp_dense and tp_dense > tp_moe))
if tp_moe and tp_dense and not tp_dense > tp_moe:
    print("Note: qwen-dense did not decode more slowly than qwen-moe this time. Not graded; your result may differ, explain it in Q1.")
for m in CHAT_MODELS:
    tag = m.replace("qwen-", "")
    for col in ["ttft_s", "tpot_s", "total_s"]:
        v = p50(m, col)
        if v is not None:
            ck.record(f"{col.replace('_s', '')}_p50_{tag}", v)

### A3. Prefill vs decode: compute-bound vs memory-bandwidth-bound

The chapter's "Computational bottlenecks" discussion is the key to reading your numbers:

- **Prefill** processes all prompt tokens at once. That is big matrix math, so it is **compute-bound**: speed depends on how many FLOPs (floating-point operations: the multiplications and additions) the chip can do per second. More active parameters means more FLOPs per prompt token.
- **Decode** makes one token at a time. For each token the chip must read the active weights from memory, but does very little math with them, so decode is **memory-bandwidth-bound**: speed depends on how many bytes must be read per token and how fast memory delivers them.

That explains the MoE advantage. `qwen-moe` has 35B parameters in total, but its router only activates about **3B per token**. `qwen-dense` activates all 27B for every token. Fewer bytes read per token means faster decode, even though the MoE model is bigger on disk.

**Course cluster at a glance** (you'll use this table in A3, A4 and the analysis):

| Alias | Model | Precision on the cluster | Weights in memory | Runs on |
|---|---|---|---|---|
| `qwen-moe` | Qwen3.6-35B-A3B (MoE, 35B total / ~3B active) | 8-bit | ~36 GB | Mac #1 and Mac #2 (one copy each, load-balanced) |
| `qwen-dense` | Qwen3.8-27B (dense) | 4-bit | ~16 GB | Mac #3 |
| `gemma-moe` | Gemma 4 26B-A4B (MoE, ~25B total / ~4B active) | 4-bit | ~16.5 GB | Mac #3 |
| `qwen-small` | Qwen3.5-4B (dense) | 8-bit | ~5 GB | Mac #3 |
| `embed` | Qwen3-Embedding-0.6B | 16-bit | ~1.2 GB | Mac #3 |
| `rerank` | Qwen3-Reranker-0.6B | 4-bit | ~0.3 GB | Mac #3 |

Each Mac Studio M3 Ultra: 96 GB unified memory, about 72 GB usable by the GPU by default, about 819 GB/s memory bandwidth, up to 8 requests at once per Mac. Mac #3 holds about 40 GB of weights, the rest is room for KV cache. Weights sizes are approximate (the "~36 GB" includes the extra multi-token-prediction layers).

**Precision matters too: bytes vs math.** `qwen-dense` is stored at 4 bits per weight and `qwen-moe` at 8 bits (quantization, from Lab 02). Fewer bits means fewer *bytes* to read for every output token, so 4-bit storage helps decode. It does not reduce the *math*: every weight still takes part in the same number of multiplications, so prefill (compute-bound) still pays for all 27B of `qwen-dense`'s active parameters. Keep this in mind for A4: decode and prefill can show different gaps between the two models.

The chapter defines **MBU (model bandwidth utilization)**: the memory bandwidth a model actually uses during decode divided by the hardware's peak bandwidth.

`MBU = (bytes read per token x tokens per second) / peak bandwidth`

An M3 Ultra has about **819 GB/s** of unified-memory bandwidth (Apple's spec). In Lab 2 you turned this into a "speed limit" (bandwidth / bytes of active weights); now you compare it with what you actually measured. `MODEL_SPECS` below gives each model's size as served (from the cluster table). The MBU cell is written for you: it computes the gigabytes read per token (active parameters x bytes per parameter), the theoretical decode ceiling (peak bandwidth / GB per token), and MBU from your measured decode speed. It ignores KV-cache reads, so it is a rough, lower-bound estimate of bytes per token.

**TODO (A3), predict first:** using only the cluster table (active parameters and precision), which of the three models should have the **highest** decode ceiling, and which the **lowest**? Fill in `PREDICT_A3` *before* you run the MBU cell; the check cell then compares your prediction with the table.

In [ ]:
# TODO: BEFORE running the MBU cell, fill in two of MOE, DENSE, SMALL (or their names in quotes)
PREDICT_A3 = {"highest_ceiling": None, "lowest_ceiling": None}

In [ ]:
PEAK_BW_GB_S = 819   # M3 Ultra unified memory bandwidth (Apple spec)

# Sizes as served on the cluster (see Lab 2 for the memory math). bytes_per_param = file size / total params.
MODEL_SPECS = {
    MOE:   {"total_params_b": 35, "active_params_b": 3,  "served_gb": 36},   # ~8-bit (oQ8e)
    DENSE: {"total_params_b": 27, "active_params_b": 27, "served_gb": 16},   # ~4-bit (oQ4e)
    SMALL: {"total_params_b": 4,  "active_params_b": 4,  "served_gb": 5},    # ~8-bit
}

mbu_rows = []
for m, s in MODEL_SPECS.items():
    bytes_per_param = s["served_gb"] / s["total_params_b"]
    measured_decode = 1 / p50(m, "tpot_s") if p50(m, "tpot_s") else None
    gb_per_token = s["active_params_b"] * bytes_per_param
    ceiling_tok_s = PEAK_BW_GB_S / gb_per_token
    mbu = gb_per_token * measured_decode / PEAK_BW_GB_S if measured_decode else None
    mbu_rows.append({"model": m, "GB_read_per_token": gb_per_token, "ceiling_tok_s": ceiling_tok_s,
                     "measured_decode_tok_s": measured_decode, "MBU": mbu})
df_mbu = pd.DataFrame(mbu_rows)
display(df_mbu.round(3))

In [ ]:
valid = set(MODEL_SPECS)
hi_pred, lo_pred = PREDICT_A3.get("highest_ceiling"), PREDICT_A3.get("lowest_ceiling")
ok_pred = hi_pred in valid and lo_pred in valid and hi_pred != lo_pred
ck.check("a3_prediction", ok_pred, 1, f"Fill in PREDICT_A3 with two different models from {sorted(valid)}.")
actual_hi = df_mbu.loc[df_mbu["ceiling_tok_s"].idxmax(), "model"]
actual_lo = df_mbu.loc[df_mbu["ceiling_tok_s"].idxmin(), "model"]
print(f"Highest ceiling: you predicted {hi_pred}, the table says {actual_hi}")
print(f"Lowest ceiling:  you predicted {lo_pred}, the table says {actual_lo}")
ck.record("a3_prediction_right", bool(ok_pred and hi_pred == actual_hi and lo_pred == actual_lo))
ck.check("mbu_measured", bool(pd.to_numeric(df_mbu["MBU"], errors="coerce").notna().all()), 1,
         "MBU needs your measured decode speed for all three models: finish A2 first, then re-run the MBU cell.")
for _, row in df_mbu.iterrows():
    if pd.notna(row["MBU"]):
        ck.record(f"mbu_{row['model'].replace('qwen-', '')}", float(row["MBU"]))
print("MBU near or above 1.0 is possible here: multi-token prediction (the '-mtp' builds) and batching with")
print("classmates' requests let one weight read serve more than one token. Discuss this in the analysis.")

**Reading the MBU table:** one row per model. `GB_read_per_token` is how many gigabytes of active weights the chip must read for each output token. `ceiling_tok_s` is the fastest decode the memory bandwidth allows (819 GB/s divided by that). `measured_decode_tok_s` is what you actually got (from your median TPOT). `MBU` is measured bytes per second divided by peak bandwidth: 0.5 means the model used about half of the hardware's memory bandwidth.

**Why can MBU come out near or above 1.0?** The formula assumes every output token costs one full read of the active weights. Two things break that assumption here. With multi-token prediction (the `-mtp` builds, Lab 02), the model guesses a few tokens ahead and checks them in one pass, so one read of the weights can produce 2 or more tokens. With batching, a classmate's request can share your read. Either way the formula counts that one read more than once, so MBU looks inflated. In the other direction, the formula ignores the KV cache, which also has to be read for every token, so the true bytes per token are a little higher than `GB_read_per_token`. That is why it is only a rough estimate.

### A4. TTFT grows with prompt length (prefill cost)

RAG (retrieval-augmented generation, Lab 08: looking up relevant documents and pasting them into the prompt) puts retrieved documents into the prompt, so prompt length is a design decision with a latency price. Here you send prompts of about **500, 2,000 and 8,000 tokens** built from Larkspur documents and watch TTFT. You ask for only 32 output tokens because you care about prefill here.

Two details:
- We build prompts by characters (about 4 per token) and read the true size from `usage.prompt_tokens`.
- oMLX keeps a **prompt cache**: if a new prompt starts with the same tokens as a recent one, it reuses that work and TTFT drops. That would ruin this experiment, so `long_prompt()` puts a random `nonce()` (a short random tag, used once) at the very **start** of every prompt. (Lab 13 turns this around and uses the cache on purpose.)

You compare `qwen-moe` (3B active) with `qwen-dense` (27B active). Prefill is compute-bound, and A3 told you what 4-bit storage does and does not save. **Predict first:** will the gap between the two models be bigger in TTFT (prefill) than it was in TPOT (decode), or smaller? Write it down before you run the sweep.

**TODO:** in the loop, send the prompt and keep the result with its target size.

In [ ]:
docs = sorted(load_docs(), key=lambda d: d["doc_id"])
CORPUS_TEXT = "\n\n".join(f"## {d['title']} ({d['doc_id']} v{d.get('version', '')})\n{d['text']}" for d in docs)
print(f"{len(docs)} documents, {len(CORPUS_TEXT):,} characters available")


def long_prompt(target_tokens):
    """About target_tokens of Larkspur text, with a unique tag first so no cached prefix can be reused."""
    context = CORPUS_TEXT[: target_tokens * 4]
    return [
        {"role": "system", "content": f"Request {nonce()}. You are Larkspur Ridge IT's help-desk assistant."},
        {"role": "user", "content": f"Reference documents:\n{context}\n\nIn one sentence, what company are these documents from?"},
    ]


TARGETS = [500, 2000, 8000]
SWEEP_MODELS = [MOE, DENSE]
SWEEP_REPS = 3

In [ ]:
sweep_records = []
for rep in range(SWEEP_REPS):
    for target in TARGETS:
        for model in SWEEP_MODELS:
            # TODO: rec = await measure_stream(...) with long_prompt(target) and max_tokens=32,
            #       then store the target size in the record under the key "target_tokens".
            rec = None
            if rec is not None:
                sweep_records.append(rec)
            await asyncio.sleep(PAUSE_S + target / 4000)   # longer prompts, longer pause (token-per-minute limit)
    print(f"sweep rep {rep + 1}/{SWEEP_REPS} done")
df_sweep = pd.DataFrame(sweep_records)
print("errors:", int((~df_sweep["ok"]).sum()) if len(df_sweep) else "no data")

In [ ]:
prefill_rate = {}
if len(df_sweep) and df_sweep["ok"].any():
    display(p50_p90(df_sweep, ["model", "target_tokens"], ["prompt_tokens", "ttft_s", "tpot_s"]).round(3))
    fig, ax = plt.subplots(figsize=(7, 3.6))
    for m in SWEEP_MODELS:
        g = df_sweep[(df_sweep.model == m) & df_sweep.ok]
        x, y = g["prompt_tokens"].astype(float), g["ttft_s"].astype(float)
        ax.scatter(x, y, color=COLORS[m], s=36, label=m)
        if len(g) >= 3 and x.nunique() > 1:
            slope, intercept = np.polyfit(x, y, 1)           # seconds per prompt token, fixed overhead
            prefill_rate[m] = 1 / slope if slope > 0 else None  # prompt tokens processed per second
            xs = np.linspace(x.min(), x.max(), 20)
            ax.plot(xs, slope * xs + intercept, color=COLORS[m], lw=2)
    ax.set_xlabel("prompt tokens (usage.prompt_tokens)"); ax.set_ylabel("TTFT (s)")
    ax.set_title("TTFT vs prompt length (lines = linear fit)"); ax.grid(alpha=0.3); ax.legend()
    plt.show()
    print("Estimated prefill speed (prompt tokens per second):", {k: round(v) for k, v in prefill_rate.items() if v})
else:
    print("Complete the TODO above first.")

**Reading the table and plot:** one row per model and target size. `prompt_tokens_p50` is the real prompt size the server counted, and the TTFT and TPOT columns show median (`_p50`) and p90 timings at that size. In the plot, each dot is one request: prompt length across, TTFT up. The straight line is a **linear fit**, the best straight line through the dots. Its slope is the extra seconds each additional prompt token costs, so `1 / slope` is the prefill speed printed under the plot.

In [ ]:
def median_ttft(model, target):
    g = df_sweep[(df_sweep.model == model) & (df_sweep.target_tokens == target) & df_sweep.ok] if len(df_sweep) else []
    return g["ttft_s"].median() if len(g) else None

ok_sweep = len(df_sweep) >= 15 and df_sweep["ok"].sum() >= 15 and df_sweep["prompt_tokens"].max() >= 5000
ck.check("sweep_ran", bool(ok_sweep), 2,
         "Run all 3 sizes x 2 models x 3 reps; the 8K prompt should report >= 5,000 prompt tokens.")
lo, hi = median_ttft(MOE, 500), median_ttft(MOE, 8000)
ck.check("ttft_grows_with_prompt", bool(lo and hi and hi > lo), 2,
         "TTFT at ~8K tokens should be larger than at ~500. Did the nonce stay at the START of the prompt?")
ck.check("prefill_rate", all(prefill_rate.get(m) for m in SWEEP_MODELS), 1,
         "The linear fit needs successful sweep data for both models.")
for m, v in prefill_rate.items():
    if v:
        ck.record(f"prefill_tok_s_{m.replace('qwen-', '')}", float(v))

## Part B - Applied lab: concurrency, throughput and goodput

### B1. Your SLO, and how one "burst" is measured

So far every request had the model (almost) to itself. Real users overlap. Chapter 9 ("Throughput and goodput") separates:

- **Throughput:** output tokens per second (or requests per second) the *system* delivers.
- **Latency:** what each *user* waits.
- **Goodput:** requests per second that **meet your service-level objective (SLO)**. Throughput that misses the SLO is not useful work.

The servers use **continuous batching**: oMLX accepts up to **8 concurrent requests per Mac** and decodes them together, so one weight read serves several users (decode is bandwidth-bound, remember). `qwen-moe` runs on **two** Macs, and the gateway picks one at random for each request. `qwen-dense`, `qwen-small`, `gemma-moe` and the embedding and reranker models all share **one** Mac (#3; see the cluster table in A3). Your key may run **4 requests in parallel**.

We measure with **bursts**: `n` requests sent at the same instant (think: `n` technicians press Enter at once), then a pause. Each request summarizes a different Larkspur runbook (about 1,000 prompt tokens, 150 output tokens), with a nonce first so the prompt cache does not flatter the numbers.

**TODO 1:** define your SLO. A reasonable starting point: TTFT under 2 s and total latency under 15 s for a 150-token answer. Justify yours in the analysis.

In [ ]:
# TODO: set your SLO (seconds). Example: {"ttft_s": 2.0, "total_s": 15.0}
SLO = {"ttft_s": None, "total_s": None}

In [ ]:
def meets_slo(rec, slo):
    """True if a request succeeded and met every limit in the SLO."""
    if not rec.get("ok") or any(v is None for v in slo.values()):
        return False
    return all(rec.get(k) is not None and rec[k] <= limit for k, limit in slo.items())

ck.check("slo_defined", all(isinstance(v, (int, float)) and v > 0 for v in SLO.values()), 1,
         "Set both numbers in SLO (seconds).")
ck.record("slo_ttft_s", SLO.get("ttft_s")); ck.record("slo_total_s", SLO.get("total_s"))

RUNBOOKS = [d for d in docs if d["folder"] == "runbooks"]


def workload_messages(i):
    """Request i of a burst: summarize one Larkspur runbook (about 1,000 prompt tokens)."""
    d = RUNBOOKS[i % len(RUNBOOKS)]
    return [
        {"role": "system", "content": f"Request {nonce()}. You are Larkspur Ridge IT's help-desk assistant."},
        {"role": "user", "content": f"Runbook {d['doc_id']}:\n{d['text'][:4000]}\n\n"
                                    "Summarize this runbook for a new technician in about 150 words."},
    ]

**TODO 2:** in `run_burst`, start all `n` requests **at the same time** with `asyncio.gather` (it starts several `await`-able calls together and waits until all of them are done). (`await measure_stream(...)` in a loop would send them one after another, which is exactly what we are *not* measuring.)

`burst_metrics` is written for you. It turns one burst into three system-level numbers: **throughput** (output tokens of the successful requests divided by the burst's wall-clock time), **goodput** (requests that met your SLO, per second of wall time) and **SLO pass rate** (the share of *sent* requests that met the SLO, so errors count as misses).

**TODO 3, predict first:** before you run the self-test, guess `PREDICT_BURST_WALL`: if one request takes about `T` seconds alone, about how many seconds will a burst of 2 *parallel* requests take, as a multiple of `T` (for example 1.0 means "no slower than one request", 2.0 means "as slow as two in a row")? The self-test prints the measured ratio next to your guess.

In [ ]:
async def run_burst(model, n, max_tokens=150, offset=0):
    """Send n requests at the same moment. Returns (list of records, wall-clock seconds for the burst)."""
    t0 = time.perf_counter()
    # TODO: recs = await asyncio.gather(*[measure_stream(model, workload_messages(offset + i), max_tokens) for i in range(n)])
    recs = []
    wall = time.perf_counter() - t0
    return list(recs), wall


def burst_metrics(recs, wall_s, slo):
    """System-level numbers for one burst."""
    ok = [r for r in recs if r["ok"]]
    good = [r for r in ok if meets_slo(r, slo)]
    tokens = sum(r["completion_tokens"] for r in ok)
    out = {"n_sent": len(recs), "n_ok": len(ok), "n_errors": len(recs) - len(ok), "wall_s": wall_s}
    out["throughput_tok_s"] = tokens / wall_s if wall_s > 0 else None
    out["goodput_req_s"] = len(good) / wall_s if wall_s > 0 else None
    out["slo_pass_rate"] = len(good) / len(recs) if recs else None
    return out

In [ ]:
# TODO: BEFORE the self-test, your guess: wall time of a 2-request parallel burst, as a multiple of one request's time
PREDICT_BURST_WALL = None

In [ ]:
# Self-test with one burst of 2 (2 requests).
test_recs, test_wall = await run_burst(MOE, 2)
test_bm = burst_metrics(test_recs, test_wall, SLO)
print(test_bm)
if len(test_recs) == 2:
    sum_total = sum(r["total_s"] or 0 for r in test_recs)
    print(f"burst wall {test_wall:.2f} s vs sum of the two latencies {sum_total:.2f} s (parallel => wall < sum)")
ck.check("burst_is_parallel", len(test_recs) == 2 and test_wall < 0.9 * sum(r["total_s"] or 0 for r in test_recs), 2,
         "run_burst must send both requests at once with asyncio.gather; wall time should be well below the sum.")
single = [x["total_s"] for x in test_recs if x.get("ok") and x.get("total_s")]
if len(single) == 2:
    print(f"your guess: {PREDICT_BURST_WALL} x one request; measured: {test_wall / max(single):.2f} x the slower request")
ck.check("burst_prediction", isinstance(PREDICT_BURST_WALL, (int, float)) and PREDICT_BURST_WALL > 0, 1,
         "Put your guess (a positive number, for example 1.5) in PREDICT_BURST_WALL before the self-test.")
ck.record("predict_burst_wall", PREDICT_BURST_WALL);

**Reading the output:** the first line is `burst_metrics()` for the test burst. `n_sent`, `n_ok` and `n_errors` count requests; `wall_s` is the clock time for the whole burst; `throughput_tok_s` is output tokens per second across both requests; `goodput_req_s` is requests per second that met your SLO; `slo_pass_rate` is the share of sent requests that met it (1.0 means all). The second line compares the burst's wall time with the two latencies added up: if the requests really ran in parallel, the wall time is much smaller than the sum. The last line puts your `PREDICT_BURST_WALL` guess next to the measured ratio (burst wall time divided by the slower request's own latency).

### B2. Concurrency sweep: 1, 2 and 4 parallel requests

This cell sends bursts at concurrency 1, 2 and 4 to `qwen-moe` and `qwen-dense` (68 requests, about 6-8 minutes). It pauses between bursts to respect the rate limit. Run it **once**. Before you run it, write down your prediction: will per-request latency go up a lot, a little, or not at all from 1 to 4? Will throughput go up?

In [ ]:
LEVELS = [1, 2, 4]
BURSTS_PER_LEVEL = {1: 8, 2: 5, 4: 4}      # 8 + 10 + 16 = 34 requests per model
CONC_MODELS = [MOE, DENSE]
BURST_PAUSE_S = 4

conc_records, burst_rows = [], []
sweep_start = dt.datetime.now(dt.timezone.utc)
for model in CONC_MODELS:
    for n in LEVELS:
        for b in range(BURSTS_PER_LEVEL[n]):
            recs, wall = await run_burst(model, n, offset=b * n)
            for rr in recs:
                rr.update(concurrency=n, burst=b)
            conc_records += recs
            burst_rows.append({"model": model, "concurrency": n, "burst": b, **burst_metrics(recs, wall, SLO)})
            await asyncio.sleep(BURST_PAUSE_S)
        print(f"{model}: concurrency {n} done")
df_conc, df_bursts = pd.DataFrame(conc_records), pd.DataFrame(burst_rows)
print(f"Sweep ran {sweep_start:%H:%M}-{dt.datetime.now(dt.timezone.utc):%H:%M} UTC; requests sent so far: {BUDGET['sent']}")

In [ ]:
level_table = pd.DataFrame()
if len(df_conc) and df_conc["ok"].any():
    lat = p50_p90(df_conc, ["model", "concurrency"], ["ttft_s", "total_s", "tpot_s"])
    sysm = (df_bursts.groupby(["model", "concurrency"])
            .agg(throughput_tok_s=("throughput_tok_s", "median"), goodput_req_s=("goodput_req_s", "median"),
                 slo_pass_rate=("slo_pass_rate", "mean"), errors=("n_errors", "sum")).reset_index())
    level_table = lat.merge(sysm, on=["model", "concurrency"])
    display(level_table.round(3))

    fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
    for m in CONC_MODELS:
        g = level_table[level_table.model == m]
        axes[0].plot(g.concurrency, g.total_s_p50, "-o", color=COLORS[m], lw=2, ms=8, label=f"{m} p50")
        axes[0].plot(g.concurrency, g.total_s_p90, "--o", color=COLORS[m], lw=2, ms=8, alpha=0.6, label=f"{m} p90")
        axes[1].plot(g.concurrency, g.ttft_s_p50, "-o", color=COLORS[m], lw=2, ms=8, label=m)
        axes[2].plot(g.concurrency, g.throughput_tok_s, "-o", color=COLORS[m], lw=2, ms=8, label=m)
    axes[1].axhline(SLO["ttft_s"] or 0, color="#888888", ls=":", lw=1); axes[1].text(1, SLO["ttft_s"] or 0, " SLO", color="#555555")
    for ax, t, y in zip(axes, ["Total latency per request", "TTFT (p50)", "System throughput"],
                        ["seconds", "seconds", "output tokens / s"]):
        ax.set_title(t); ax.set_xlabel("concurrent requests"); ax.set_ylabel(y)
        ax.set_xticks(LEVELS); ax.grid(alpha=0.3); ax.set_ylim(bottom=0); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print("No concurrency data yet; complete the TODOs in B1.")

**Reading the table and plots:** one row per model and concurrency level (1, 2 or 4 requests at once). The `ttft_s`, `total_s` and `tpot_s` columns are what each *user* experienced (median `_p50` and `_p90`). `throughput_tok_s` and `goodput_req_s` are what the *system* delivered per burst (median over the bursts), `slo_pass_rate` is the share of requests that met your SLO, and `errors` counts failed requests. In the plots, concurrency is across. Left: total latency per request (solid line p50, dashed p90). Middle: median TTFT, with your SLO as a dotted line. Right: system throughput. Compare how fast the left two rise with how fast the right one rises.

Throughput and goodput here are **per burst**: tokens (or good requests) divided by the time one burst took. The pauses between bursts are not counted, so these are not the sustained capacity of the service; B4 estimates that.

In [ ]:
def lt(model, n, col):
    g = level_table[(level_table.model == model) & (level_table.concurrency == n)] if len(level_table) else []
    return float(g[col].iloc[0]) if len(g) and pd.notna(g[col].iloc[0]) else None

ck.check("concurrency_all_levels", len(level_table) == len(CONC_MODELS) * len(LEVELS), 2,
         "Expected rows for both models at concurrency 1, 2 and 4. Check df_conc['error'] for failures.")
t1, t4 = lt(MOE, 1, "throughput_tok_s"), lt(MOE, 4, "throughput_tok_s")
# Not graded (it depends on the cluster, not on your code; burst_is_parallel already checked your code).
ck.record("throughput_scales", bool(t1 and t4 and t4 > 1.3 * t1))
if t1 and t4 and not t4 > 1.3 * t1:
    print("Note: throughput at 4 parallel was not clearly above 1 this time. Not graded; explain it in Q3 (busy cluster?).")
for m in CONC_MODELS:
    tag = m.replace("qwen-", "")
    for n in LEVELS:
        for col, short in [("throughput_tok_s", "thru"), ("total_s_p50", "lat_p50"), ("goodput_req_s", "goodput")]:
            v = lt(m, n, col)
            if v is not None:
                ck.record(f"{short}_{tag}_c{n}", v)

**Slowest requests (for Q4):** the next cell lists the five slowest requests of the sweep and which Mac served each one (`mac-A`, `mac-B`, ...; `unknown` if the gateway didn't say). Compare their `sent_utc` times and Macs: did the slow ones cluster at one moment or on one Mac?

In [ ]:
if len(df_conc) and df_conc["ok"].any():
    ok_conc = df_conc[df_conc.ok].copy()
    ok_conc["mac"] = label_backends(ok_conc)
    display(ok_conc.sort_values("total_s", ascending=False).head(5)[["model", "concurrency", "mac", "ttft_s", "tpot_s", "total_s", "sent_utc"]].round(2))
else:
    print("Run the concurrency sweep first.")

### B3. Optional: the 8-way test (only if your instructor raised the limit)

Your key normally allows 4 parallel requests. If you send 8, the gateway rejects the extra ones with HTTP error 429 ("Too Many Requests") instead of queuing them. During this week your instructor may raise the limit to 8 so you can see what happens when one Mac's 8 slots fill up. The next cell asks the gateway what your key's limit is. Set `RUN_8WAY = True` **only** if it says 8 or more (or your instructor announced it). It adds 16 requests.

In [ ]:
import httpx
key_limit = None
try:
    info = httpx.get(base_url.rstrip("/").removesuffix("/v1") + "/key/info",
                     headers={"Authorization": f"Bearer {api_key}"}, timeout=15).json()
    key_limit = (info.get("info") or {}).get("max_parallel_requests")
except Exception as e:
    print("Could not read key info:", type(e).__name__)
print("Your key's max_parallel_requests:", key_limit)

RUN_8WAY = False     # set True only if the limit above is 8 or more
if RUN_8WAY:
    recs8 = []
    for b in range(2):
        recs, wall = await run_burst(MOE, 8, offset=b * 8)
        bm = burst_metrics(recs, wall, SLO)
        print(f"8-way burst {b + 1}: {bm}")
        for rr in recs:
            rr.update(concurrency=8, burst=b)
        recs8 += recs
        await asyncio.sleep(8)
    df8 = pd.DataFrame(recs8)
    display(p50_p90(df8, "concurrency", ["ttft_s", "total_s", "tpot_s"]).round(3))
    print("errors:", df8["error"].dropna().str[:60].value_counts().to_dict())
    ck.record("ran_8way", True)

### B4. Capacity estimate: how many Larkspur technicians?

Larkspur has 16 active technicians. Could this cluster serve all of them, at your SLO, if the assistant ran on `qwen-moe`?

Use **Little's law**: `average concurrency = arrival rate x average latency`. If the system can hold `C_MAX` requests at once while still meeting the SLO, and each request takes `L` seconds, it can accept `C_MAX / L` requests per second. If each technician asks one question every `T` seconds at peak, the number of technicians is `C_MAX / L x T`.

**An everyday picture.** Think of a help desk with phones. With 4 phones (`C_MAX = 4`) and calls that take 2 minutes each (`L`), the desk finishes at most 4 / 2 = 2 calls per minute. If each technician calls once every 10 minutes (`T`), 2 calls per minute x 10 minutes = 20 technicians can share the 4 phones before callers start waiting. The code below does the same arithmetic in seconds. (Little's law is a general rule from queueing theory, not from the textbook.)

**One assumption to know about:** the code uses the latency you measured at `C_MAX`. You only measured 1, 2 and 4 parallel requests, so for any other value it uses the latency at 4. Because latency grows with concurrency (B2), a `C_MAX` above 4 gives an **optimistic** estimate; the cell prints a warning when that happens.

**TODO:** choose `C_MAX` and `SECONDS_BETWEEN_QUESTIONS` and justify both in the analysis. For `C_MAX`, start from what you measured (did 4-way still meet your SLO?), then think about the hardware: 2 Macs x 8 slots = 16 slots for `qwen-moe`, shared with your classmates.

In [ ]:
# TODO: your assumptions (numbers). Justify them in the analysis.
C_MAX = None                       # concurrent requests the cluster holds while meeting your SLO
SECONDS_BETWEEN_QUESTIONS = None   # how often one technician asks the assistant at peak (seconds)

In [ ]:
def capacity(c_max, latency_s, seconds_between_questions):
    """Little's law: requests/s the system accepts = c_max / latency; technicians = that x seconds between questions."""
    return c_max / latency_s * seconds_between_questions

L_LEVEL = C_MAX if C_MAX in LEVELS else 4          # we only measured 1, 2 and 4
L = lt(MOE, L_LEVEL, "total_s_p50")
techs = None
if C_MAX and SECONDS_BETWEEN_QUESTIONS and L:
    techs = capacity(C_MAX, L, SECONDS_BETWEEN_QUESTIONS)
    print(f"L = {L:.2f} s at concurrency {L_LEVEL}  ->  about {techs:.0f} technicians")
    if C_MAX not in LEVELS:
        print(f"Warning: you did not measure concurrency {C_MAX}, so this uses the latency measured at 4."
              + (" Latency grows with concurrency, so this estimate is optimistic." if C_MAX > 4 else ""))
    print(f"If all 16 qwen-moe slots met the SLO at the same latency (optimistic): about {capacity(16, L, SECONDS_BETWEEN_QUESTIONS):.0f}")
ck.check("capacity_estimate", bool(techs and techs > 0), 1, "Set C_MAX and SECONDS_BETWEEN_QUESTIONS, and finish B2.")
if techs:
    ck.record("capacity_technicians", float(techs)); ck.record("capacity_c_max", C_MAX)
ck.record("requests_sent", BUDGET["sent"])
print("Requests this notebook sent:", BUDGET["sent"])

In [ ]:
# Save your raw measurements (handy for the analysis; not required for submission).
pd.concat([df_a.assign(part="A2"), df_sweep.assign(part="A4"), df_conc.assign(part="B2")], ignore_index=True) \
  .drop(columns=["text", "chunk_times"], errors="ignore").to_csv("lab12_measurements.csv", index=False)
print("Saved lab12_measurements.csv")

## Written analysis

Answer each question in a few short paragraphs. Use **your own numbers** from the tables above, and cite Chapter 9 by section name or concept where asked.

**Q1. Prefill, decode, and the MoE advantage.** Using your A2 table (median and p90 TTFT and TPOT for all three models) and your A3 MBU table: explain why `qwen-moe` decodes faster than `qwen-dense` even though it has more total parameters. Use the compute-bound vs memory-bandwidth-bound distinction from Chapter 9 ("Computational bottlenecks") and the MBU definition ("Utilization, MFU, and MBU"). If any MBU came out near or above 1.0, explain what could cause that. Where does `qwen-small` fit, and is it the fastest decoder? Why or why not? (Compare `GB_read_per_token` in your A3 table.)

*Your answer:*

**Q2. Prompt length.** From your A4 sweep, report TTFT at about 500, 2K and 8K tokens for both models, and your estimated prefill speeds. Why does TTFT grow with prompt length, and why is the gap between `qwen-moe` and `qwen-dense` larger in prefill than you might expect from decode? What does this mean for how many retrieved chunks Larkspur's RAG assistant (Lab 8) should put in each prompt?

*Your answer:*

**Q3. Throughput vs latency.** Using your B2 table and plots: how did per-request latency (p50 and p90) and system throughput change from 1 to 2 to 4 parallel requests, for each model? Explain the trade-off using Chapter 9's "Throughput and goodput" section (it notes that techniques like batching can improve throughput but reduce latency; the book means latency gets worse, because requests wait longer) and the batching explanation in B1. Why does `qwen-moe` (two Macs, chosen at random per request) behave differently from `qwen-dense` (one Mac shared with `qwen-small`, `gemma-moe` and the embedding model)? What was your goodput at concurrency 4, and what would you predict for an 8-way test, given that each Mac batches at most 8 requests? (If you ran B3, use those numbers.)

*Your answer:*

**Q4. Noise.** When did you run this lab (day and local time; the notebook recorded UTC)? How large was the spread between p50 and p90 in your measurements, and which parts of the system could cause it (classmates, the load balancer, the network, the VPN)? Use the slowest-requests table after B2: pick a request that was much slower than the rest and suggest a cause. If you had to give Larkspur numbers you could defend, how would you change the benchmark?

*Your answer:*

**Q5. Capacity for Larkspur.** Explain your `C_MAX` and `SECONDS_BETWEEN_QUESTIONS` choices and your capacity estimate. Could this cluster serve Larkspur's 16 technicians at your SLO? What would break first if usage grew 10x: TTFT, TPOT, or the gateway's rate limits? (Hint: when every slot is busy, new requests wait in a queue before prefill even starts. Which of your metrics would include that wait?) Name one measurement you would add before promising a number to the Service Desk Manager.

*Your answer:*

## AI-use reflection

Answer briefly but specifically.

1. What did you use AI assistance for in this lab (tools, which steps)?

*Your answer:*

2. Where did the AI help most, and where was it wrong or unhelpful? How did you notice?

*Your answer:*

3. What is one thing you now understand that you could explain without AI help?

*Your answer:*

4. Lab-specific: did an AI assistant ever suggest timing code that would measure the wrong thing (for example `time.time()` around a non-streamed call, counting chunks as tokens, or a loop that ran 'parallel' requests one at a time)? How did you catch it, or how would you?

*Your answer:*

## Submit

Run this last, save, upload the .ipynb.

In [ ]:
ck.report();